# Decision-Calibrated FNO World Models
## From predictive uncertainty to decision-effective dynamics ambiguity sets

This portable Jupyter workflow keeps four claims separate:

1. **FNO prediction:** a residual Fourier Neural Operator learns one-step PDE dynamics.
2. **Uncertainty:** clean/perturbed FNO disagreement supplies a local scale.
3. **Calibration:** split conformal scores turn that scale into field-valued ambiguity sets.
4. **Control:** robust MPC is evaluated independently; coverage alone does not imply control benefit or safety.

The notebook uses project-relative paths and ordinary Python subprocesses. It can run in JupyterLab, VS Code, or another Python notebook environment.

## 1. Locate the repository and inspect the runtime

Launch this notebook from anywhere inside a clone of the repository. The cell walks upward until it finds `pyproject.toml`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the project repository.')


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Project:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)

## 2. Install the project when needed

Set `INSTALL_PROJECT=True` in a fresh environment. The optional `ns2d` dependency installs the official NeuralOperator dataset loader.

In [ ]:
INSTALL_PROJECT = False

if INSTALL_PROJECT:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', '.[dev,ns2d]'],
        check=True,
    )
else:
    print('Using the current Python environment.')

## 3. Command helper

All experiments are called as Python modules, so the notebook does not depend on notebook shell syntax.

In [ ]:
def run_module(module: str, *arguments: object) -> None:
    command = [sys.executable, '-m', module, *map(str, arguments)]
    print(' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## 4. CPU-compatible FNO smoke test

This verifies data generation, residual FNO training, perturbation calibration, and MPC without reproducing the full paper table.

In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    run_module(
        'unoc.experiment',
        '--model', 'fno',
        '--uncertainty', 'perturbation',
        '--quick',
        '--device', DEVICE,
        '--seed', 27,
        '--output-dir', 'results/notebook_smoke',
    )

## 5. Full controlled-Burgers experiment

The full run trains a clean FNO and a perturbed-label FNO for 60 epochs, evaluates four deployment regimes, and compares six MPC variants on 24 matched actuator-gain cases. A GPU is recommended but not required by the code.

In [ ]:
RUN_FULL_BURGERS = False
BURGERS_OUTPUT = PROJECT_ROOT / 'results' / 'fno_burgers_seed27_reproduction'

if RUN_FULL_BURGERS:
    run_module(
        'unoc.experiment',
        '--model', 'fno',
        '--uncertainty', 'perturbation',
        '--control-cases', 24,
        '--control-horizon', 20,
        '--device', DEVICE,
        '--seed', 27,
        '--output-dir', BURGERS_OUTPUT,
    )

## 6. Value-gap and value-bound experiments

These experiments require a trained perturbation-FNO checkpoint. The value-gap experiment tests deterministic scaling separately from conformal coverage. The bound comparison uses 60 calibration and 160 independent test trajectories.

In [ ]:
RUN_VALUE_AUDITS = False
CHECKPOINT = BURGERS_OUTPUT / 'fno_perturbation_world_model.pt'

if RUN_VALUE_AUDITS:
    if not CHECKPOINT.exists():
        raise FileNotFoundError('Run the full Burgers experiment first.')
    run_module(
        'unoc.value_gap',
        '--checkpoint', CHECKPOINT,
        '--output-root', 'experiments/value_gap_reproduction',
    )
    run_module(
        'unoc.bound_comparison',
        '--checkpoint', CHECKPOINT,
        '--output-root', 'experiments/bound_comparison_reproduction',
        '--calibration-cases', 60,
        '--test-cases', 160,
        '--horizon', 20,
        '--gamma', 0.95,
    )

## 7. Optional NS2D FNO uncertainty benchmark

The official NeuralOperator loader retrieves Zenodo record `12825163`. The 128×128 archive is approximately 1.5 GB. It contains vorticity input-output fields with no action channel, so this experiment tests field uncertainty rather than closed-loop control.

In [ ]:
RUN_NS2D = False
NS2D_DATA = PROJECT_ROOT / 'data' / 'ns2d'

if RUN_NS2D:
    run_module(
        'unoc.ns2d_experiment',
        '--data-root', NS2D_DATA,
        '--output-dir', 'experiments/ns2d_reproduction',
        '--n-train', 800,
        '--n-audit', 200,
        '--n-test', 300,
        '--epochs', 20,
        '--batch-size', 8,
        '--modes', 16,
        '--hidden-channels', 32,
        '--label-noise', 0.05,
        '--smoothing-window', 15,
        '--seed', 27,
    )

## 8. Audit the released numbers

The audit recomputes the controller statistics, coverage denominators, value-bound table, theorem slopes, and NS2D coverage-width row from the saved source files.

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/audit_release_results.py'],
    cwd=PROJECT_ROOT,
    check=True,
)
print((PROJECT_ROOT / 'results/data_integrity/DATA_AUDIT.md').read_text()[:2000])

## 9. Inspect standalone figures

Every file carries one conclusion. The following cell displays the released PNG figures without modifying them.

In [ ]:
from IPython.display import display
from PIL import Image

figure_roots = [
    PROJECT_ROOT / 'results/fno_burgers_seed27/figures',
    PROJECT_ROOT / 'experiments/value_gap_reference/figures',
    PROJECT_ROOT / 'experiments/bound_comparison_reference/figures',
    PROJECT_ROOT / 'experiments/ns2d_reference/figures',
]
for figure_root in figure_roots:
    for path in sorted(figure_root.glob('*.png')):
        print(path.relative_to(PROJECT_ROOT))
        display(Image.open(path))